# Phase 1 — ColBERT recall probe (R8 gate)

Pretrained-first (no fine-tuning): index the A1-**enriched** 47k docs with an off-the-shelf
late-interaction model (PyLate PLAID), retrieve for dev with a **focused** query (recent
utterance + goal, R8 §4.4), and measure **recall@{50,100,200,500}** overall + cold/warm and
**unique recall** vs the current fused pool. The decisive question: does late interaction surface
golds the current channels (BM25 + dense + cknn + cf + artist) miss — i.e. raise the pool ceiling
(~0.65 today)? Gate: real unique recall / fused lift → wire into R7 fusion; below dense → consider
Phase-B fine-tune. Spec: `47_R8_colbert_late_interaction_channel.md`.

## 1. Drive + HF auth (Colab Secrets)

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
import os
DRIVE='/content/drive/MyDrive/recsys2026'
os.environ['HF_HOME']=f'{DRIVE}/hf_cache'; OUT=f'{DRIVE}/outputs'
os.makedirs(os.environ['HF_HOME'],exist_ok=True); os.makedirs(OUT,exist_ok=True)
try:
    t=userdata.get('HF_TOKEN'); os.environ['HF_TOKEN']=os.environ['HUGGINGFACE_HUB_TOKEN']=t
    from huggingface_hub import login; login(t); print('HF ok')
except Exception as e: print('no HF_TOKEN secret:', e)

## 2. Clone + install (pylate for ColBERT)

In [ ]:
!git clone --branch fresh-start --depth 1 https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026 2>/dev/null || (cd /content/recsys2026 && git pull)
%cd /content/recsys2026
!pip -q install datasets bm25s scipy scikit-learn lightgbm sentence-transformers numpy pandas pylate
import sys; sys.path.insert(0,'.')

## 3. Config

In [ ]:
# ColBERT (pretrained, off-the-shelf). GTE-ModernColBERT: ModernBERT backbone, long ctx, strong BEIR.
COLBERT_MODEL='lightonai/GTE-ModernColBERT-v1'   # fallback: 'colbert-ir/colbertv2.0'
Q_LEN=64               # focused query token cap (recent utterance + goal is short)
D_LEN=300              # enriched-doc token cap (trim raw tags, keep doc2query — R8 §4.3)
BSIZE=128
MAXK=500; KS=[50,100,200,500]
FOCUSED_QUERY=True     # ColBERT gets recent-utterance+goal (R8 §4.4); existing channels get the full query
DEV_SESSIONS=1000
IDX_FOLDER=f'{OUT}/colbert_plaid'; IDX_NAME='gte-moderncolbert-enriched-d%d' % D_LEN

# retrieval setup (shared with phase2_rerank — keep identical so the baseline pool matches)
DENSE_MODEL='BAAI/bge-large-en-v1.5'
DENSE_QUERY_PREFIX='Represent this sentence for searching relevant passages: '
CONTENT_MODALITIES={'cknn_audio':'audio-laion_clap','cknn_attr':'attributes-qwen3_embedding_0.6b'}
ORG='talkpl-ai'
ENRICHED_GLOB=f'{OUT}/catalog_enriched_*.parquet'
print('ColBERT:', COLBERT_MODEL, '| Q_LEN', Q_LEN, '| D_LEN', D_LEN, '| focused', FOCUSED_QUERY)

## 4. Load enriched catalog + dev + channels + fusion (the baseline pool)

In [ ]:
import glob, os, pickle
import pandas as pd
from datasets import load_dataset
from mcrs.data.catalog import Catalog
from mcrs.data.embeddings import TrackEmbeddings, UserEmbeddings
from mcrs.data.conversations import Conversations
from sentence_transformers import SentenceTransformer
from mcrs.retrieval.query import QueryBuilder
from mcrs.retrieval.bm25_channel import BM25Channel
from mcrs.retrieval.dense_channel import DenseChannel
from mcrs.retrieval.personalization import ContentKNNChannel, CFChannel, SameArtistChannel
from mcrs.retrieval.related_artist import RelatedArtistChannel, build_artist_cooc, tid_to_artists_from_catalog
from mcrs.retrieval.fusion import RRFFusion

meta_rows=load_dataset(f'{ORG}/TalkPlayData-Challenge-Track-Metadata',split='all_tracks')
_enr=sorted(glob.glob(ENRICHED_GLOB))
assert _enr, f'No enriched parquet at {ENRICHED_GLOB} — run A1 first (ColBERT indexes enriched docs)'
_edf=pd.read_parquet(_enr[-1]); enr=dict(zip(_edf['track_id'],_edf['enriched_doc']))
cat=Catalog(meta_rows,enriched_docs=enr); USE_ENRICHED=True
print(f'enriched docs {len(enr)} (from {_enr[-1].split("/")[-1]})')

tre=load_dataset(f'{ORG}/TalkPlayData-Challenge-Track-Embeddings',split='all_tracks')
_avail=set(tre.column_names)
CKNN_MODS={lab:mod for lab,mod in CONTENT_MODALITIES.items() if mod in _avail}
te={lab:TrackEmbeddings(tre.select_columns(['track_id',mod]),modalities=[mod]) for lab,mod in CKNN_MODS.items()}
te_cf=TrackEmbeddings(tre.select_columns(['track_id','cf-bpr']),modalities=['cf-bpr'])
ued=load_dataset(f'{ORG}/TalkPlayData-Challenge-User-Embeddings'); ue=UserEmbeddings([r for sp in ued for r in ued[sp]])
dsd=load_dataset(f'{ORG}/TalkPlayData-Challenge-Dataset')
conv_dv=Conversations(dsd['test'].select(range(DEV_SESSIONS)))

model=SentenceTransformer(DENSE_MODEL,device='cuda')
doc_mat=model.encode([cat.id_to_metadata(t,enriched=USE_ENRICHED) for t in cat.index_to_id],batch_size=256,normalize_embeddings=True,show_progress_bar=True)
dense=DenseChannel(cat.index_to_id,doc_mat,lambda qs:model.encode([DENSE_QUERY_PREFIX+q for q in qs],batch_size=256,normalize_embeddings=True),normalize=False)

COOC_PKL=f'{OUT}/artist_cooc.pkl'
if os.path.exists(COOC_PKL):
    cooc=pickle.load(open(COOC_PKL,'rb')); print('loaded cooc',len(cooc),'artists')
else:
    cooc=build_artist_cooc(dsd['train'],tid_to_artists_from_catalog(cat)); pickle.dump(cooc,open(COOC_PKL,'wb')); print('built cooc',len(cooc))

cknn=[ContentKNNChannel(te[lab],mod,label=lab) for lab,mod in CKNN_MODS.items()]
chans=[BM25Channel(cat,enriched=USE_ENRICHED), dense, *cknn,
       CFChannel(ue,te_cf,'cf-bpr'), SameArtistChannel(cat), RelatedArtistChannel(cat,cooc)]
fusion=RRFFusion(chans,k=60); print('baseline channels:',[c.label for c in chans])

## 5. Build the ColBERT PLAID index over enriched docs (cached to Drive)

In [ ]:
from pylate import indexes, models, retrieve
from mcrs.retrieval.colbert_channel import colbert_doc_text   # enriched doc, track_id prefix stripped

cmodel=models.ColBERT(model_name_or_path=COLBERT_MODEL, query_length=Q_LEN, document_length=D_LEN)
_index_path=os.path.join(IDX_FOLDER, IDX_NAME)
if os.path.exists(_index_path):
    index=indexes.PLAID(index_folder=IDX_FOLDER, index_name=IDX_NAME, override=False)
    print('loaded cached PLAID index <-', _index_path)
else:
    doc_ids=list(cat.index_to_id)
    doc_texts=[colbert_doc_text(cat, t) for t in doc_ids]
    print(f'encoding {len(doc_ids)} enriched docs with {COLBERT_MODEL} (one-time)...')
    doc_emb=cmodel.encode(doc_texts, batch_size=BSIZE, is_query=False, show_progress_bar=True)
    index=indexes.PLAID(index_folder=IDX_FOLDER, index_name=IDX_NAME, override=True)
    index.add_documents(documents_ids=doc_ids, documents_embeddings=doc_emb)
    print('built + cached PLAID index ->', _index_path)
cb_retriever=retrieve.ColBERT(index=index)

## 6. Retrieve (focused query) + recall / unique-recall probe

In [ ]:
from mcrs.data.ids import canonical_track_id
from mcrs.eval.probe import recall_ceiling

dv=list(conv_dv.turns())
qb=QueryBuilder()

def focused(t):
    parts=[t.utterances[-1] if t.utterances else '']
    if t.goal: parts.append(t.goal)
    return ' '.join(p for p in parts if p)

cb_queries=[focused(t) for t in dv] if FOCUSED_QUERY else [qb.build(t).text for t in dv]
print('ColBERT-encoding', len(cb_queries), 'dev queries + retrieving top', MAXK, '...')
q_emb=cmodel.encode(cb_queries, batch_size=BSIZE, is_query=True, show_progress_bar=True)
cb_results=cb_retriever.retrieve(queries_embeddings=q_emb, k=MAXK)
colbert_lists=[[canonical_track_id(h['id']) for h in q] for q in cb_results]

# existing channels get the FULL query (each channel its best-suited input — R8 §4.4)
queries=[qb.build(t).text for t in dv]
bc=[{'history_tids':t.history_tids,'user_id':t.user_id} for t in dv]; uids=[t.user_id for t in dv]
per_channel={ch.label: ch.batch_text_to_item_retrieval(queries, MAXK, bc, uids) for ch in fusion.channels}
per_channel['colbert']=colbert_lists

golds=[conv_dv.gold(t.session_id,t.turn_number) for t in dv]
segments=[t.segment for t in dv]
rep_all =recall_ceiling(per_channel, golds, KS, segments)                                  # incl ColBERT
rep_base=recall_ceiling({k:v for k,v in per_channel.items() if k!='colbert'}, golds, KS, segments)  # current pool
print('probe done over', len(dv), 'dev turns')

## 7. Gate verdict

In [ ]:
import json
def r(d): return {k: round(d[k],4) for k in KS}
cb=rep_all['per_channel']['colbert']; dn=rep_all['per_channel']['dense']
print('ColBERT recall      :', r(cb['recall']))
print('  cold              :', r(cb['by_segment']['cold']))
print('  warm              :', r(cb['by_segment']['warm']))
print('ColBERT unique_recall (sole hitter across all channels):', round(cb['unique_recall'],4))
print('dense   recall      :', r(dn['recall']), '| dense unique:', round(dn['unique_recall'],4))
print('FUSED  w/o ColBERT  :', r(rep_base['fused']['recall']))
print('FUSED  w/  ColBERT  :', r(rep_all['fused']['recall']))
lift={k: round(rep_all['fused']['recall'][k]-rep_base['fused']['recall'][k],4) for k in KS}
print('FUSED recall LIFT   :', lift)

ships = lift[200] > 0.002 or cb['unique_recall'] > 0.01
print('\nGATE:', 'PROMISING — wire into R7 fusion (+ consider replacing R4 query-dense)' if ships
      else 'WEAK — pretrained adds little; consider Phase-B fine-tune or drop')
json.dump({'colbert':cb,'dense':dn,'fused_base':rep_base['fused'],'fused_all':rep_all['fused'],
           'lift':lift,'model':COLBERT_MODEL,'focused':FOCUSED_QUERY},
          open(f'{OUT}/phase1_colbert_probe.json','w'), indent=2)
print('saved ->', f'{OUT}/phase1_colbert_probe.json')

## 8. Next

- **Promising** (fused recall lift @200/@500 or real unique recall): wire ColBERT into R7 fusion as
  a channel (weight-swept), and run the replace-R4 ablation (R8 §6). Then re-probe the fused ceiling.
- **Weak** (≈ dense, no unique recall): the pretrained model doesn't transfer to music CRS — that's
  the signal for Phase-B fine-tune on Train query→gold pairs (`salvage/scripts/train_colbert.py`),
  gated. Don't fine-tune before this gate.
- Try `FOCUSED_QUERY=False` (full query) and `colbert-ir/colbertv2.0` as ablations if the focused
  GTE-ModernColBERT result is borderline.